# Loan Approval Classifier

Quick supervised classification. Features: income, credit score, loan amount, term, employment status. Target: approved (1) or not (0). The dataset is synthetic — 500 rows generated with plausible-looking correlations so the pipeline is end-to-end runnable.

Two models: logistic regression (because it's the right starting point for a small tabular dataset) and a random forest for comparison.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

np.random.seed(1)
df = pd.read_csv("../../data/loan_approval/loan_approval.csv")
print(df.shape, "approval rate:", df.approved.mean().round(3))
df.head()


(500, 6) approval rate: 0.502


,income,credit_score,loan_amount,term_months,employed,approved
0,63944,724,32996,72,1,1
1,81345,569,8746,120,1,0
2,67875,721,26180,36,1,1
3,151318,725,33626,60,1,1
4,78850,732,16216,48,1,1


In [2]:
X = df.drop(columns=["approved"]).values.astype(float)
y = df["approved"].values
features = df.drop(columns=["approved"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)
sc = StandardScaler().fit(X_train)
X_train_s, X_test_s = sc.transform(X_train), sc.transform(X_test)

lr = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, random_state=1, n_jobs=-1).fit(X_train, y_train)

for name, model, Xte in [("LogReg", lr, X_test_s), ("RandomForest", rf, X_test)]:
    proba = model.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"--- {name} ---  AUC={roc_auc_score(y_test, proba):.3f}")
    print(classification_report(y_test, pred))


--- LogReg ---  AUC=0.791
              precision    recall  f1-score   support

           0       0.72      0.76      0.74        50
           1       0.74      0.70      0.72        50

    accuracy                           0.73       100
   macro avg       0.73      0.73      0.73       100
weighted avg       0.73      0.73      0.73       100

--- RandomForest ---  AUC=0.778
              precision    recall  f1-score   support

           0       0.72      0.82      0.77        50
           1       0.79      0.68      0.73        50

    accuracy                           0.75       100
   macro avg       0.75      0.75      0.75       100
weighted avg       0.75      0.75      0.75       100



In [3]:
# coefficient signs from logreg should make basic sense
coef = pd.Series(lr.coef_[0], index=features).sort_values()
print(coef.round(3))


loan_amount    -0.399
term_months    -0.281
income          0.110
employed        0.312
credit_score    0.765
dtype: float64


## Note on fairness

Loan approval is one of those domains where the *modeling* part is the easy part. The hard part is what features you're allowed to use, what disparate-impact testing looks like, and whether the historical labels themselves are biased. This notebook does none of that. It's a baseline classifier on synthetic data, nothing more.

For an actual fintech project I'd at minimum:
- Run group-fairness checks (demographic parity, equalized odds) across protected attributes if available.
- Avoid using ZIP code or other proxies for race.
- Use a model with built-in explanations (SHAP) since applicants in many jurisdictions have a right to an explanation for an adverse decision.
